# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step exploration of the FAIR² Mental Health Survey Dataset, leveraging the [mlcroissant](https://mlcommons.org/croissant/) library to load, inspect, and analyze its content. Each dataset entity, such as record sets and fields, is referenced by its `@id`. The analysis demonstrates common data handling and visualization routines suitable for AI-ready datasets.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")  # Suppress typical pandas warnings for demo clarity

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print dataset high-level metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published on: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and structure.

> In Croissant, a **RecordSet** represents a logically tabular collection (like a DataFrame). Each RecordSet contains **Fields** (which may map to columns), all of which have unique `@id`s. Let's enumerate all record sets and show their fields by `@id`.

In [ ]:
# Gather all RecordSet @id(s) defined in the schema
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in the dataset metadata.\n(Beta notice: Some datasets may not use the explicit 'recordSet' key; mlcroissant will find them if present in the Croissant schema.)")
else:
    print("## Record sets and their fields in the dataset:\n")
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        if hasattr(rs, 'fields') and rs.fields:
            for field in rs.fields:
                print(f"  Field @id: {field.id:40} name: {getattr(field,'name', '(no name)')}")
        else:
            print("  No fields listed.")
        print("")

# For later usage, collect a sample RecordSet @id
record_set_ids = [rs.id for rs in record_sets] if record_sets else []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

> We use the primary survey record set for this example.
You may change the `selected_record_set` below if you wish to analyze a different record set.

In [ ]:
# If no record set IDs were detected try the fallback (first available record set, if any)
if len(record_set_ids) == 0 and hasattr(dataset, 'record_sets'):
    record_set_ids = [rs.id for rs in dataset.record_sets if hasattr(rs, 'id')]

# Use the first record set for exploration (customize if needed)
if record_set_ids:
    selected_record_set = record_set_ids[0]
    print(f"Loading records from record set: {selected_record_set}")
else:
    raise ValueError("No record sets found to extract data from.")

df = pd.DataFrame(dataset.records(record_set=selected_record_set))
print(f"Loaded DataFrame with shape {df.shape} for record set {selected_record_set}")
print("Column names:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records, normalizing numeric fields, transforming distributions, and grouping data. We select fields using their Croissant `@id`s.

> The following operations will:
> - Filter by a numeric survey score (e.g., `cr:psq_total_score` if present),
> - Normalize it (zero mean, unit std),
> - Group by a demographic field (e.g., `cr:gender` or `cr:level_of_education`),
> All field references are by `@id`.

In [ ]:
# Choose a field for numeric analysis (update according to actual field @ids if necessary)
# The following @id are typical for mental health scales; adjust as appropriate for your schema
possible_numeric_fields = [
    'cr:psq_total_score',      # Perceived Stress Questionnaire
    'cr:phq9_total_score',    # PHQ-9 depression scale
    'cr:gad7_total_score'     # GAD-7 anxiety scale
]
numeric_field = None
for field in possible_numeric_fields:
    if field in df.columns:
        numeric_field = field
        break
if not numeric_field:
    raise ValueError("Could not detect a numeric field suitable for demonstration from the expected list.")

# Filter for high scores (arbitrary threshold, adapt as needed)
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} records")
display_cols = [numeric_field]
print(filtered_df[display_cols].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to group by a demographic field
group_fields_priority = ['cr:gender', 'cr:level_of_education', 'cr:age', 'cr:village']
group_field = None
for f in group_fields_priority:
    if f in filtered_df.columns:
        group_field = f
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nAverage {numeric_field} grouped by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable demographic field found to group data by.")

## 5. Visualization

Visualize data distributions and group-wise differences. Here, we show a histogram and a boxplot for the selected numeric field, and if applicable, a bar plot by demographic group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot for the numeric field (for entire dataset)
plt.figure(figsize=(6, 5))
sns.boxplot(x=df[numeric_field].dropna())
plt.title(f"Boxplot of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Group-wise mean barplot (if group_field available)
if group_field:
    plt.figure(figsize=(8, 5))
    sns.barplot(x=group_field, y=numeric_field, data=filtered_df, ci=None)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.show()


## 6. Conclusion
This notebook demonstrated a full workflow for loading, inspecting, processing, and visualizing the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using the `mlcroissant` library.

**Key findings:**
- The dataset is organized according to the Croissant schema and exposes record sets and fields with unique `@id`s.
- Specific survey scores (e.g., GAD-7, PHQ-9, PSQ) were processed and visualized, revealing their distributions and demographic patterns (when available).
- The Croissant metadata approach allows reproducible, schema-driven AI-ready dataset processing.

> Next steps: For domain-specific research, consider more detailed analyses (multivariate, longitudinal, factor analysis) or integration with public health benchmarks using field `@id`s for robust feature engineering.